# Week 8: Listing Summarization and Answerability

Generate concise listing summaries, evaluate ROUGE-L, and explain unsupported or empty searches. BART/T5 fine-tuning is optional and is not implemented.

In [1]:
from pathlib import Path
import sys
import importlib.util

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/query_parser.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the IDX Exchange NLP project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

missing = [name for name in ("nltk", "pandas", "ipywidgets", "rouge_score")
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "This notebook is using the wrong or incomplete Python environment. "
        "Select kernel 'IDX Exchange NLP (.venv)' (or the project's .venv/bin/python), "
        "then run this cell again. Missing: " + ", ".join(missing) +
        ". Current Python: " + sys.executable
    )

from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/query_parser.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Markdown
from scripts.listing_summarizer import ListingSummarizer
from scripts.answerability import AnswerabilityChecker, search_and_summarize
from scripts.build_week8_artifacts import TEST_REFERENCE_PATH, run_pipeline
summarizer = ListingSummarizer()
sample = pd.read_csv(ROOT / "data/processed/listing_sample.csv")
cases = json.loads(TEST_REFERENCE_PATH.read_text())["cases"]
print(f"Local listings: {len(sample):,}; independent test records: {len(cases)}")

Local listings: 1,000; independent test records: 20


## 1. Main deliverable: original-sentence extraction

The main `summary` selects up to three original sentences. It scores explicit room counts, price, location and supported features, then rewards new information and penalizes repeated content, boilerplate and excessive length. Feature-list scores are capped so kitchen details do not overwhelm bed/bath counts. Selected sentences retain their source order. Local NLTK Punkt rules work offline. Empty input generates no invented prose; descriptions with fewer than two useful sentences are reported as exceptions.

In [2]:
record = cases[0]["record"]
print("SOURCE REMARKS:\n", record["remarks"])
print("\nMAIN EXTRACTIVE SUMMARY:\n", summarizer.summarize_listing(record)["summary"])

SOURCE REMARKS:
 West Palmdale home with 3 bedrooms, 2 bathrooms and a spacious bonus room for entertainment or home office. The living room features a vaulted ceiling and a brick fireplace, connecting to the dining area and kitchen via a breakfast bar. The open floor plan creates a spacious interior environment. The sliding door opens onto the outdoor patio and spacious backyard. The property has an 8,019 sq ft lot which provides limitless opportunities—ideal for entertaining, gardening, or potential enhancements like a pool, ADU, or an outdoor lounge area. There is also a two-car detached garage with a long driveway. Located near shopping, dining, and the 14 freeway, this home is ready for its next owner or as an investment property.

MAIN EXTRACTIVE SUMMARY:
 West Palmdale home with 3 bedrooms, 2 bathrooms and a spacious bonus room for entertainment or home office. The property has an 8,019 sq ft lot which provides limitless opportunities—ideal for entertaining, gardening, or potent

## 2. Trace selected sentences and keep MLS fields separate

`summary` and `extractive_summary` contain the same original sentences. Listed price, city and numeric fields remain separate in `facts`. They are never inserted into the main prose. `selected_sentences` records source indices, scoring roles and scores. `missing_fields` identifies information not covered by the selected sentences; `not_in_remarks` identifies information not detected anywhere in the remarks. These are heuristic diagnostics, not proof of absence. Historical rent is not treated as listed price. Potential room-count conflicts remain metadata warnings. The old template appears only as `template_summary` for comparison.

In [3]:
examples = [summarizer.summarize_listing(case["record"]) for case in cases[:5]]
display(pd.DataFrame([{k: row[k] for k in ["listing_id", "summary", "top_features", "missing_fields", "warnings"]} for row in examples]))
display(pd.DataFrame(examples[0]["feature_evidence"]))

,listing_id,summary,top_features,missing_fields,warnings
0,1156538875,"West Palmdale home with 3 bedrooms, 2 bathroom...","[detached garage, driveway]",[price],[]
1,1189473380,"This charming 4-bedroom, 2-bath home offers a ...",[attached garage],"[price, feature_2]",[]
2,1176603062,"In Toluca Lake, the jump from two bedrooms to ...",[primary suite],"[baths, price, feature_2]",[]
3,1174225864,Located in the highly sought-after Temecula Be...,"[pool, spa]",[price],[]
4,1174744255,This striking recently-renovated home has turn...,"[pool, gym]","[beds, baths, price]",[]


,term,category,sentence,sentence_index,score
0,detached garage,exterior_feature,There is also a two-car detached garage with a...,2,9.4
1,driveway,exterior_feature,There is also a two-car detached garage with a...,2,4.2


In [4]:
conflicts = [summarizer.summarize_listing(case["record"]) for case in cases if summarizer.summarize_listing(case["record"])["warnings"]]
if conflicts:
    display(pd.DataFrame(conflicts)[["listing_id", "summary", "facts", "warnings"]].head(3))
else:
    print("No explicit early numeric room-count discrepancies in this test sample.")

,listing_id,summary,facts,warnings
0,1183472536,Welcome to this beautifully maintained single-...,"{'beds': 4, 'baths': 3, 'price': 928000, 'loca...",[baths: listing field 3 differs from early rem...
1,1159976751,"The main home offers 3 bedrooms, 2 bathrooms, ...","{'beds': 4, 'baths': 3, 'price': 309000, 'loca...",[beds: listing field 4 differs from early rema...


In [5]:
details = summarizer.summarize_listing(record)
display(pd.DataFrame(details["selected_sentences"]))
print("Separate MLS fields:", details["facts"])
print("Not detected in remarks:", details["not_in_remarks"])
print("Template comparison only:", details["template_summary"])

,index,text,roles,score,word_count
0,0,"West Palmdale home with 3 bedrooms, 2 bathroom...","[baths, beds, feature:bonus room, feature:home...",17.400000,18
1,4,"The property has an 8,019 sq ft lot which prov...","[feature:adu, size]",3.100000,27
2,5,There is also a two-car detached garage with a...,"[feature:detached garage, feature:driveway, pr...",4.483333,11


Separate MLS fields: {'beds': 3, 'baths': 2, 'price': 425000, 'location': 'Palmdale'}
Not detected in remarks: ['price']
Template comparison only: This home in Palmdale has 3 bedrooms and 2 bathrooms and is listed at $425,000. Highlights include a detached garage and a fireplace.


## 3. ROUGE evaluation and dataset provenance

The development cohort contains 20 listings (seed 8). Because all 40 earlier development/test records had been inspected, the revised test cohort contains 20 different listings (seed 20261115), excluding all 40. References are independently assistant-selected sentences from original remarks, selected before inspecting revised test predictions. They do not insert MLS prices or room counts absent from remarks. They still require human confirmation. Do not tune on this test cohort.

Compare the previous sentence scorer at budgets 2 and 3, the revised main selector and the old template against the **same** source-only references. Report macro-average ROUGE-1/2/L precision, recall and F1 using `rouge-score`, with Porter stemming. The target is **mean test ROUGE-L F1 > 0.4**.  Source-selected references naturally reward sentence overlap; this does not establish factual correctness or replace human review.

In [6]:
report = run_pipeline()
score_table = pd.DataFrame(report["scores"]).T
display(score_table[["rouge1_fmeasure", "rouge2_fmeasure", "rougeL_precision", "rougeL_recall", "rougeL_fmeasure"]].round(4))
print("Reference status:", report["evaluation_status"])
print("Human-verified references:", report["human_verified_references"], "/", report["reference_count"])
print("Main extractive summary target met numerically:", report["primary_rouge_target_met_numerically"])
print("All summaries match source sentences:", report["all_summaries_are_source_sentences"])
print("Fewer than two useful source sentences:", report["source_sentences_under_two"])
print("Ratings:", report["human_evaluation"])
print("Saved summaries:", report["summary_count"])

,rouge1_fmeasure,rouge2_fmeasure,rougeL_precision,rougeL_recall,rougeL_fmeasure
baseline_2_sentences,0.5981,0.5366,0.6902,0.5266,0.5732
baseline_3_sentences,0.6599,0.5992,0.6416,0.6662,0.6343
extractive,0.6130,0.5457,0.6113,0.5839,0.5767
template_comparison_only,0.2621,0.0667,0.3196,0.1297,0.1804


Reference status: provisional_draft_references
Human-verified references: 0 / 20
Main extractive summary target met numerically: True
All summaries match source sentences: True
Fewer than two useful source sentences: 4
Ratings: {'required': 20, 'completed': 20, 'status': 'complete'}
Saved summaries: 1000


## 4. Answerability before query execution

Reuse the Week 4 parser and validation. Accept property searches even when they omit words such as “home” but specify a recognized city. Reject unsupported questions, invalid cities/values and suspicious SQL fragments with specific messages. These rules describe the supported database-search scope; a supported query can still produce zero matches.

In [7]:
checker = AnswerabilityChecker()
queries = ["Find 3+ bedroom homes in Irvine under $1M with a pool", "Irvine under $1M", "Find homes in mars", "Predict home prices next year", "Homes with school ratings in Irvine", "Find 99 bedroom homes in Irvine", "Homes; DROP TABLE rets_property", "Tell me a bedtime story"]
display(pd.DataFrame([{"query": q, **checker.assess_query(q)} for q in queries])[["query", "answerable", "reason", "message"]])

,query,answerable,reason,message
0,Find 3+ bedroom homes in Irvine under $1M with...,True,supported_search,The query can be searched using the available ...
1,Irvine under $1M,True,supported_search,The query can be searched using the available ...
2,Find homes in mars,False,invalid_city,City 'mars' is not in the available city list.
3,Predict home prices next year,False,unsupported_question,The listing database cannot predict future pri...
4,Homes with school ratings in Irvine,False,unsupported_question,The current search schema does not provide ver...
5,Find 99 bedroom homes in Irvine,False,invalid_filters,beds_min must be between 0 and 20
6,Homes; DROP TABLE rets_property,False,unsafe_query,Please enter a property search without SQL com...
7,Tell me a bedtime story,False,out_of_domain,This does not appear to be a supported real-es...


## 5. Answerability after execution

Distinguish zero matches, all-empty content, useful results, and an unavailable database. A connection failure is not evidence that no listings match.

In [8]:
post_cases = {"no matches": pd.DataFrame(), "null content": pd.DataFrame([{"L_ListingID": "123", "remarks": None, "price": None}]), "useful results": sample.head(2), "database unavailable": None}
display(pd.DataFrame([{"example": name, **checker.assess_results(frame)} for name, frame in post_cases.items()]))

,example,answerable,reason,message
0,no matches,False,no_matches,No listings match your criteria. Try relaxing ...
1,null content,False,empty_content,The query returned listings without meaningful...
2,useful results,True,results_found,Found 2 matching listings.
3,database unavailable,False,data_unavailable,No database response is available.


## 6. End-to-end offline demonstration

This example searches the local Week 1 CSV using parsed filters, then runs the same post-check and summarizer as the database workflow. It does not contact MySQL. The production function defaults to the existing parameterized Week 4 MySQL executor.

In [9]:
def local_search(query, *, limit, parser):
    f = parser.parse(query)
    frame = sample.copy()
    if f.city:
        frame = frame[frame.L_City.str.casefold() == f.city.casefold()]
    for attr, col, comparison in [("price_min", "price", "ge"), ("price_max", "price", "le"), ("beds_min", "beds", "ge"), ("beds_max", "beds", "le"), ("baths_min", "baths", "ge"), ("baths_max", "baths", "le")]:
        value = getattr(f, attr)
        if value is not None:
            frame = frame[getattr(frame[col], comparison)(value)]
    for term in f.amenities_include + f.terms_include:
        frame = frame[frame.remarks.fillna("").str.contains(term, case=False, regex=False)]
    for term in f.amenities_exclude + f.terms_exclude:
        frame = frame[~frame.remarks.fillna("").str.contains(term, case=False, regex=False)]
    return frame.head(limit)
response = search_and_summarize("Homes in Irvine", checker=checker, search_function=local_search, limit=3)
print(response["message"])
display(pd.DataFrame(response["results"])[["listing_id", "summary"]])

Found 3 matching listings.


,listing_id,summary
0,1176984096,Prime exterior-facing location for this highly...
1,1176830956,Two additional bedrooms are complemented by a ...
2,1190477353,"Beautifully maintained home offering comfort, ..."


## 7. Teammate evaluation: 20 extractive summaries

Open `outputs/week8/week8_summary_review.xlsx`. Main Summary contains the new extractive output. Original remarks and a separate draft Reference summary are included. Rate factual accuracy, completeness against information actually present in remarks, and readability from 1 to 5, and enter a reviewer name. Do not penalize extraction for omitting a price absent from remarks; listed price remains separate metadata. Check negation, partial counts, material repair warnings and missing useful source content. Approve or correct the draft reference separately. Blank ratings are pending.

Import the completed workbook with:
```bash
python scripts/import_week8_review.py /path/to/week8_summary_review.xlsx
```
The importer verifies IDs and current Summary text, so ratings from the old template cannot count as ratings of the revised extractive summary. Old files remain in `outputs/week8/archive/`. Ratings and reference approvals remain separate.

### Interactive review widget

Run the next cell to review the same 20 test samples one at a time. Previous, Next and the Sample selector save your current draft; use Save review before closing. The original remarks, main extractive summary and separate MLS fields are visible. Enter your name, three 1–5 ratings and notes. Blank ratings remain pending. The reference-review panel and template comparison are optional to expand.

Drafts are stored in `data/processed/week8_widget_reviews.json` and reload automatically. Click **Update evaluation** to import the saved reviews through the same validation as Excel and refresh the canonical report. Summary ratings do not automatically approve a reference. Excel remains an alternative; avoid editing both at the same time. No database is required.

In [16]:
from pathlib import Path
import sys
import importlib.util

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/query_parser.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the IDX Exchange NLP project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

missing = [name for name in ("nltk", "pandas", "ipywidgets", "rouge_score")
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "This notebook is using the wrong or incomplete Python environment. "
        "Select kernel 'IDX Exchange NLP (.venv)' (or the project's .venv/bin/python), "
        "then run this cell again. Missing: " + ", ".join(missing) +
        ". Current Python: " + sys.executable
    )

import importlib
import scripts.import_week8_review as review_io
import scripts.week8_review_widget as review_ui
importlib.reload(review_io)
importlib.reload(review_ui)

if "review_widget" in globals():
    review_widget.view.close()
review_widget = review_ui.create_week8_review_widget(root=ROOT)
print("Python kernel:", sys.executable)
review_widget.display()


Python kernel: /Users/dengyingxin/Desktop/IDX Exchange/NLP intern/.venv/bin/python


In [12]:
saved_report = json.loads((ROOT / "data/processed/week8_summary_evaluation.json").read_text())
print("Human evaluation:", saved_report["human_evaluation"])
print("Reference verification:", saved_report["human_verified_references"], "/", saved_report["reference_count"])

Human evaluation: {'required': 20, 'completed': 20, 'status': 'complete'}
Reference verification: 0 / 20


## 9. Findings and remaining work

- The main output is genuine sentence extraction. MLS facts and template prose are separate comparison metadata.
- Coverage-aware selection favors complementary room counts and feature sentences; it does not rewrite, add or silently correct original text. Missing facts, conflicting counts, malformed and very long source sentences remain limitations.
- The same-reference baselines above measure the revised selector fairly. Changing reference protocol is not an algorithmic improvement by itself.
- Test records exclude all previously inspected records, but references are still assistant-selected drafts. Numerical ROUGE is provisional until reviewed.
- Twenty teammate ratings and reference confirmation remain required. No human scores have been fabricated.
- ROUGE measures lexical overlap, not truth. Source descriptions may themselves contain incorrect or misleading claims.
- Answerability uses explicit rules and inherits Week 4 parser limits. BART/T5 fine-tuning is optional and was not performed.

ROUGE implementation: [Google Research](https://github.com/google-research/google-research/tree/master/rouge).